In [11]:
import mne
import numpy as np
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import glob
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [12]:
dataset_path = Path('/home/aloo/CNS_Summer_Project_1/Infants_data')

subjects = sorted([d for d in os.listdir(dataset_path) if d.startswith('sub-NORB')])
print(f"Found {len(subjects)} subjects")

Found 103 subjects


In [13]:
def find_edf_files(subject_id):
    """Find all EDF files for a given subject."""
    subject_path = dataset_path / subject_id
    edf_files = []
    
    if subject_path.exists():
        edf_files = list(subject_path.rglob('*.edf'))
    
    return edf_files

test_subject = subjects[68]
edf_files = find_edf_files(test_subject)
print(f"\n{test_subject}:")
print(f"  Found {len(edf_files)} EDF file(s)")
for edf in edf_files:
    print(f"  - {edf.name}")


sub-NORB00069:
  Found 4 EDF file(s)
  - sub-NORB00069_ses-2_task-EEG_eeg.edf
  - sub-NORB00069_ses-1_task-EEG_eeg.edf
  - sub-NORB00069_ses-3_task-EEG_eeg.edf
  - sub-NORB00069_ses-4_task-EEG_eeg.edf


In [14]:
def load_edf_data(edf_path):
    """
    Load EDF file with MNE and extract EEG channels.
    """
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    
    n_channels = len(raw.ch_names)
    sampling_rate = raw.info['sfreq']
    duration = raw.times[-1]
    
    print(f"\nLoaded: {Path(edf_path).name}")
    # print(f"  Channels: {n_channels}")
    # print(f"  Sampling rate: {sampling_rate} Hz")
    # print(f"  Duration: {duration:.2f} seconds")
    # print(f"  Channel names: {raw.ch_names}")
    
    return raw

if edf_files:
    raw = load_edf_data(edf_files[0])
else:
    print("No EDF files found for testing")


Loaded: sub-NORB00069_ses-2_task-EEG_eeg.edf


In [15]:
def detect_bad_channels(raw, flat_threshold=1e-12, noise_threshold=1e-2):
    """
    Detect bad channels (flat or extremely noisy).
    """
    data = raw.get_data()
    channel_names = raw.ch_names
    bad_channels = []
    
    # Define reference/ground channel names to exclude
    ref_ground_channels = {'PG1', 'PG2', 'REF', 'GND', 'GROUND', '25+', '26+', '27+'}
    
    for i, ch_name in enumerate(channel_names):
        ch_data = data[i, :]
        ch_std = np.std(ch_data)
        ch_range = np.max(ch_data) - np.min(ch_data)
        
        is_bad = False
        
        # Check if flat
        if ch_std < flat_threshold:
            is_bad = True
        
        # Check if too noisy
        elif ch_std > noise_threshold:
            is_bad = True
        
        # Check if constant value
        elif ch_range == 0:
            is_bad = True
        
        # Check for reference/ground channels
        elif ch_name.upper() in ref_ground_channels:
            is_bad = True
        
        if is_bad:
            bad_channels.append(ch_name)
    
    # print(f"\nFound {len(bad_channels)} bad channel(s): {bad_channels}")
    
    return bad_channels

if 'raw' in locals():
    bad_channels = detect_bad_channels(raw)
    
    # Remove bad channels if any found
    if bad_channels:
        raw_clean = raw.copy()
        raw_clean.drop_channels(bad_channels)
        print(f"\nRemoved {len(bad_channels)} bad channel(s)")
        print(f"Remaining channels ({len(raw_clean.ch_names)}): {raw_clean.ch_names}")
    else:
        raw_clean = raw.copy()
        print("\nNo bad channels detected. Proceeding with all channels.")
    
    raw_filtered = raw_clean  # For compatibility with downstream code
else:
    print("Error: 'raw' data not found. Please run Step 1 first.")


No bad channels detected. Proceeding with all channels.


In [16]:
def load_annotations(subject_id, session_id, dataset_path):
    """
    Load annotation file for a given subject and session.
    Handles TSV files with trailing tabs/whitespace robustly.
    """
    annotations_path = dataset_path / 'derivatives' / 'NeuronicEEG' / subject_id / session_id / 'eeg' / f'{subject_id}_{session_id}_task-EEG_annotations.tsv'
    
    if annotations_path.exists():
        # Read file line by line to handle trailing tabs properly
        with open(annotations_path, 'r') as f:
            lines = f.readlines()
        
        # Parse manually to avoid pandas issues with trailing tabs
        data = []
        header = None
        
        for i, line in enumerate(lines):
            # Split by tab and strip whitespace from each field
            fields = [field.strip() for field in line.strip().split('\t')]
            
            if i == 0:
                # Header row
                header = fields[:3]  # Only take first 3 columns: onset, duration, label
            else:
                # Data row - only take first 3 fields
                if len(fields) >= 3 and fields[0] and fields[1]:  # Ensure we have onset and duration
                    try:
                        onset = float(fields[0])
                        duration = float(fields[1])
                        label = fields[2] if fields[2] else 'eyes_closed'  # Default if empty
                        data.append([onset, duration, label])
                    except ValueError:
                        continue  # Skip malformed lines
        
        if data:
            annotations_df = pd.DataFrame(data, columns=['onset', 'duration', 'label'])
            return annotations_df
        else:
            print(f"  Warning: No valid data in annotations file: {annotations_path}")
            return None
    else:
        print(f"  Warning: No annotations file found at {annotations_path}")
        return None

def segment_data_from_annotations(raw, annotations_df, label='eyes_closed'):
    """
    Segment EEG data based on annotations file.
    Only extracts segments marked with the specified label (e.g., 'eyes_closed').
    """
    sfreq = raw.info['sfreq']
    data = raw.get_data()
    n_channels, n_samples = data.shape

    # Normalize labels for robust matching
    label_lower = label.lower().strip()
    ann_labels = annotations_df['label'].fillna('').astype(str).str.lower().str.strip()

    # Second approach: only map Spanish alias when requested label is eyes_closed
    if label_lower == 'eyes_closed':
        filtered_annotations = annotations_df[
            (ann_labels == 'eyes_closed') |
            (ann_labels == 'ojos_cerrados')
        ]
    else:
        filtered_annotations = annotations_df[ann_labels == label_lower]
    
    segments = []
    segment_info = []
    
    # print(f"\nSegmenting data from annotations file...")
    # print(f"  Total annotations: {len(annotations_df)}")
    # print(f"  Unique labels found: {annotations_df['label'].unique().tolist()}")
    # print(f"  Annotations matching '{label}' {len(filtered_annotations)}")
    
    for idx, row in filtered_annotations.iterrows():
        onset = row['onset']  # in seconds
        duration = row['duration']  # in seconds
        
        # Convert to sample indices
        start_sample = int(onset * sfreq)
        end_sample = int((onset + duration) * sfreq)
        
        # Check if segment is within bounds
        if start_sample >= 0 and end_sample <= n_samples:
            segment = data[:, start_sample:end_sample]
            segments.append(segment)
            segment_info.append({
                'onset': onset,
                'duration': duration,
                'start_sample': start_sample,
                'end_sample': end_sample
            })
        else:
            print(f"  Warning: Segment at onset={onset}s exceeds data bounds, skipping")
    
    print(f"\nExtracted {len(segments)} valid segments")
    
    return segments, segment_info

# Apply segmentation based on annotations
if 'raw_clean' in locals():
    # Extract subject and session info from the loaded EDF file
    if 'edf_files' in locals() and edf_files:
        edf_name = edf_files[0].stem
        parts = edf_name.split('_')
        subject_id = parts[0]  # e.g., 'sub-NORB00042'
        session_id = parts[1] if len(parts) > 1 and 'ses' in parts[1] else 'ses-1'
        
        # Load annotations
        annotations_df = load_annotations(subject_id, session_id, dataset_path)
        
        if annotations_df is not None:
            # Segment data based on annotations
            segments, segment_info = segment_data_from_annotations(raw_clean, annotations_df, label='eyes_closed')
            
            if len(segments) > 0:
                print(f"\n✓ Successfully segmented data for {subject_id} {session_id}")
            else:
                print(f"\n⚠ No valid segments found for {subject_id} {session_id}")
        else:
            print(f"\n⚠ Could not load annotations. Skipping segmentation.")
    else:
        print("Error: EDF file information not found.")
else:
    print("Error: 'raw_clean' data not found. Please run bad channel detection first.")


Extracted 20 valid segments

✓ Successfully segmented data for sub-NORB00069 ses-2


In [17]:
def compute_session_segment_correlation_matrices(segments, method='pearson'):

    """

    Compute one channel-by-channel correlation matrix per EEG segment.



    Parameters

    ----------

    segments : list of np.ndarray

        List of segment arrays with shape (n_channels, n_samples).

    method : str, optional

        Correlation method. Currently supports only 'pearson'.



    Returns

    -------

    correlation_matrices : list of np.ndarray

        Correlation matrix for each valid segment, each shaped (n_channels, n_channels).

    metadata : list of dict

        Per-segment metadata including index and shape.

    """

    if method.lower() != 'pearson':

        raise ValueError("Only 'pearson' method is currently supported.")



    if segments is None or len(segments) == 0:

        print("No segments provided. Returning empty results.")

        return [], []



    correlation_matrices = []

    metadata = []



    for i, segment in enumerate(segments):

        if not isinstance(segment, np.ndarray):

            print(f"Skipping segment {i}: not a NumPy array.")

            continue



        if segment.ndim != 2:

            print(f"Skipping segment {i}: expected 2D array, got shape {segment.shape}.")

            continue



        n_channels, n_samples = segment.shape

        if n_samples < 2:

            print(f"Skipping segment {i}: insufficient samples ({n_samples}).")

            continue



        # Correlate channels against each other: rows=channels, cols=time samples

        corr = np.corrcoef(segment)



        # Replace NaN values from zero-variance channels to keep matrices usable downstream

        corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)



        # Force exact self-correlation on diagonal for numerical stability

        np.fill_diagonal(corr, 1.0)



        correlation_matrices.append(corr)

        metadata.append({

            'segment_index': i,

            'n_channels': n_channels,

            'n_samples': n_samples

        })



    print(f"Computed {len(correlation_matrices)} correlation matrix/matrices from {len(segments)} segment(s).")

    return correlation_matrices, metadata





# Example usage with previously extracted session segments

if 'segments' in locals() and len(segments) > 0:

    segment_correlation_matrices, segment_corr_meta = compute_session_segment_correlation_matrices(segments)

    print(f"First matrix shape: {segment_correlation_matrices[0].shape}")

else:

    print("No 'segments' found. Run segmentation first, then rerun this cell.")


Computed 20 correlation matrix/matrices from 20 segment(s).
First matrix shape: (19, 19)


In [18]:
import networkx as nx



def compute_segment_betweenness_centrality(

    correlation_matrices,

    min_abs_corr=0.0,

    use_absolute=True,

    normalized=True,

    eps=1e-10

):

    """

    Compute node-level betweenness centrality for each segment.



    Parameters

    ----------

    correlation_matrices : list of np.ndarray

        List of square correlation matrices (n_channels x n_channels), one per segment.

    min_abs_corr : float, optional

        Minimum absolute correlation required to keep an edge.

    use_absolute : bool, optional

        If True, use |corr| as edge strength.

    normalized : bool, optional

        Whether to return normalized betweenness centrality.

    eps : float, optional

        Small value to avoid division by zero while converting strength to distance.



    Returns

    -------

    betweenness_per_segment : list of np.ndarray

        One vector per segment with node betweenness values.

    betweenness_df : pd.DataFrame

        Long-format table with columns: segment_index, node_index, betweenness.

    """

    if correlation_matrices is None or len(correlation_matrices) == 0:

        print("No correlation matrices provided. Returning empty results.")

        return [], pd.DataFrame(columns=['segment_index', 'node_index', 'betweenness'])



    betweenness_per_segment = []

    rows = []



    for seg_idx, corr in enumerate(correlation_matrices):

        if not isinstance(corr, np.ndarray):

            print(f"Skipping segment {seg_idx}: matrix is not a NumPy array.")

            continue



        if corr.ndim != 2 or corr.shape[0] != corr.shape[1]:

            print(f"Skipping segment {seg_idx}: expected square matrix, got shape {corr.shape}.")

            continue



        adj = np.array(corr, dtype=float, copy=True)

        adj = np.nan_to_num(adj, nan=0.0, posinf=0.0, neginf=0.0)

        np.fill_diagonal(adj, 0.0)



        if use_absolute:

            adj = np.abs(adj)



        if min_abs_corr > 0:

            adj[adj < min_abs_corr] = 0.0



        G = nx.from_numpy_array(adj)



        # Convert correlation strength into path distance for shortest-path betweenness.

        for u, v, d in G.edges(data=True):

            w = d.get('weight', 0.0)

            d['distance'] = 1.0 / (w + eps)



        if G.number_of_edges() == 0:

            bc_values = np.zeros(adj.shape[0], dtype=float)

        else:

            bc_dict = nx.betweenness_centrality(G, weight='distance', normalized=normalized)

            bc_values = np.array([bc_dict[i] for i in range(adj.shape[0])], dtype=float)



        betweenness_per_segment.append(bc_values)



        for node_idx, bc in enumerate(bc_values):

            rows.append({

                'segment_index': seg_idx,

                'node_index': node_idx,

                'betweenness': bc

            })



    betweenness_df = pd.DataFrame(rows)

    print(f"Computed betweenness centrality for {len(betweenness_per_segment)} segment(s).")

    return betweenness_per_segment, betweenness_df





# Example usage

if 'segment_correlation_matrices' in locals() and len(segment_correlation_matrices) > 0:

    segment_betweenness, segment_betweenness_df = compute_segment_betweenness_centrality(

        segment_correlation_matrices,

        min_abs_corr=0.0,

        use_absolute=True,

        normalized=True

    )

    print(f"Betweenness vector shape for first segment: {segment_betweenness[0].shape}")

    print(segment_betweenness_df.head())

else:

    print("No 'segment_correlation_matrices' found. Run the correlation-matrix cell first.")


Computed betweenness centrality for 20 segment(s).
Betweenness vector shape for first segment: (19,)
   segment_index  node_index  betweenness
0              0           0     0.006536
1              0           1     0.000000
2              0           2     0.000000
3              0           3     0.000000
4              0           4     0.000000


In [19]:
import re

from pathlib import Path

from matplotlib.animation import FuncAnimation, PillowWriter



def sort_nodes_fctp_order(channel_names):

    """

    Return node indices sorted by region prefix order: F, C, T, P, O.

    Remaining channels are appended at the end.

    """

    region_order = {'F': 0, 'C': 1, 'T': 2, 'P': 3, 'O': 4}



    def parse_channel(ch_name):

        name = str(ch_name).upper().strip()

        match = re.match(r'^([A-Z]+)(\d*)$', name)

        if match:

            prefix = match.group(1)

            number = int(match.group(2)) if match.group(2) else 999

        else:

            prefix = name[:1] if name else 'Z'

            number = 999



        primary = prefix[0] if prefix else 'Z'

        region_rank = region_order.get(primary, 99)

        return (region_rank, primary, number, name)



    sorted_indices = sorted(range(len(channel_names)), key=lambda i: parse_channel(channel_names[i]))

    sorted_names = [channel_names[i] for i in sorted_indices]

    return sorted_indices, sorted_names





def create_betweenness_change_gif(

    segment_betweenness,

    channel_names,

    output_path='betweenness_change_across_segments.gif',

    fps=2,

    figsize=(16, 6),

    ylim=None

):

    """

    Create a GIF showing node betweenness change across segments.



    X-axis: nodes in fixed F, C, T, P, O ordering.

    Y-axis: betweenness values.

    """

    if segment_betweenness is None or len(segment_betweenness) == 0:

        raise ValueError('segment_betweenness is empty. Compute betweenness first.')



    if channel_names is None or len(channel_names) == 0:

        raise ValueError('channel_names is empty. Provide EEG channel labels.')



    n_nodes = len(channel_names)

    for i, vec in enumerate(segment_betweenness):

        if len(vec) != n_nodes:

            raise ValueError(

                f'Segment {i} has {len(vec)} nodes, but channel_names has {n_nodes}. '

                'Ensure both come from the same cleaned recording.'

            )



    node_order_idx, ordered_node_names = sort_nodes_fctp_order(channel_names)

    ordered_betweenness = np.array([np.asarray(vec)[node_order_idx] for vec in segment_betweenness], dtype=float)



    if ylim is None:

        y_min = float(np.nanmin(ordered_betweenness))

        y_max = float(np.nanmax(ordered_betweenness))

        pad = (y_max - y_min) * 0.1 if y_max > y_min else 0.1

        ylim = (max(0.0, y_min - pad), y_max + pad)



    fig, ax = plt.subplots(figsize=figsize)

    x = np.arange(n_nodes)

    bars = ax.bar(x, ordered_betweenness[0], color='steelblue')



    ax.set_xticks(x)

    ax.set_xticklabels(ordered_node_names, rotation=90)

    ax.set_ylabel('Betweenness Centrality')

    ax.set_xlabel('Nodes (Fixed Order: F -> C -> T -> P -> O)')

    ax.set_ylim(*ylim)

    title = ax.set_title('Segment 1')

    fig.tight_layout()



    def update(frame_idx):

        values = ordered_betweenness[frame_idx]

        for bar, val in zip(bars, values):

            bar.set_height(float(val))

        title.set_text(f'Segment {frame_idx + 1}')

        return (*bars, title)



    anim = FuncAnimation(fig, update, frames=ordered_betweenness.shape[0], interval=1000 // max(1, fps), blit=False)



    output_path = Path(output_path)

    output_path.parent.mkdir(parents=True, exist_ok=True)

    anim.save(str(output_path), writer=PillowWriter(fps=fps))

    plt.close(fig)



    print(f'GIF saved to: {output_path.resolve()}')

    return output_path





# Example usage

if 'segment_betweenness' in locals() and len(segment_betweenness) > 0:

    if 'raw_clean' in locals():

        channel_names_for_plot = list(raw_clean.ch_names)

    elif 'raw_filtered' in locals():

        channel_names_for_plot = list(raw_filtered.ch_names)

    elif 'raw' in locals():

        channel_names_for_plot = list(raw.ch_names)

    else:

        raise ValueError('No EEG channel names found (raw_clean/raw_filtered/raw).')



    gif_path = create_betweenness_change_gif(

        segment_betweenness=segment_betweenness,

        channel_names=channel_names_for_plot,

        output_path='betweenness_change_across_segments.gif',

        fps=2

    )

else:

    print("No 'segment_betweenness' found. Run the betweenness computation cell first.")


GIF saved to: /home/aloo/CNS_Summer_Project_1/betweenness_change_across_segments.gif


In [20]:
from collections import defaultdict



def _parse_subject_session_from_edf(edf_path):

    """Extract (subject_id, session_id) from EDF filename or path."""

    edf_path = Path(edf_path)

    stem_parts = edf_path.stem.split('_')



    subject_id = None

    session_id = None



    for part in stem_parts:

        if part.startswith('sub-'):

            subject_id = part

        if part.startswith('ses-'):

            session_id = part



    # Fallback to directory names if filename does not include full info

    if subject_id is None:

        for p in edf_path.parts:

            if p.startswith('sub-'):

                subject_id = p

                break



    if session_id is None:

        for p in edf_path.parts:

            if p.startswith('ses-'):

                session_id = p

                break



    if subject_id is None or session_id is None:

        return None, None



    return subject_id, session_id





def find_subject_session_edf_map(dataset_path):

    """Map subject -> session -> list of EDF files."""

    dataset_path = Path(dataset_path)

    mapping = defaultdict(lambda: defaultdict(list))



    all_edf = sorted(dataset_path.rglob('*.edf'))

    for edf in all_edf:

        subject_id, session_id = _parse_subject_session_from_edf(edf)

        if subject_id and session_id:

            mapping[subject_id][session_id].append(edf)



    return mapping





def process_multi_session_subjects_betweenness(

    dataset_path,

    output_root='betweenness_multi_session',

    label='eyes_closed',

    min_abs_corr=0.0,

    gif_fps=2

):

    """

    Process all sessions for subjects that have more than one session.



    For each subject-session:

    1) load EDF

    2) clean bad channels

    3) segment by annotations

    4) compute correlation matrices

    5) compute node betweenness per segment

    6) save GIF + CSV (+ NPY) named with subject-session id

    """

    dataset_path = Path(dataset_path)

    output_root = Path(output_root)

    output_root.mkdir(parents=True, exist_ok=True)



    subject_session_map = find_subject_session_edf_map(dataset_path)

    multi_session_subjects = {

        s: sessions for s, sessions in subject_session_map.items() if len(sessions) > 1

    }



    print(f"Subjects with >1 session: {len(multi_session_subjects)}")



    summary_rows = []



    for subject_id, session_dict in sorted(multi_session_subjects.items()):

        for session_id, edf_list in sorted(session_dict.items()):

            subject_session_id = f"{subject_id}_{session_id}"

            print(f"\nProcessing {subject_session_id}...")



            if len(edf_list) == 0:

                print("  No EDF files found for this session. Skipping.")

                summary_rows.append({

                    'subject_id': subject_id,

                    'session_id': session_id,

                    'status': 'no_edf'

                })

                continue



            # Use the first EDF in the session by default

            edf_path = sorted(edf_list)[0]



            try:

                raw = load_edf_data(edf_path)



                bad_channels = detect_bad_channels(raw)

                raw_clean_local = raw.copy()

                if bad_channels:

                    raw_clean_local.drop_channels(bad_channels)



                annotations_df = load_annotations(subject_id, session_id, dataset_path)

                if annotations_df is None:

                    print("  Missing/invalid annotations. Skipping.")

                    summary_rows.append({

                        'subject_id': subject_id,

                        'session_id': session_id,

                        'status': 'no_annotations'

                    })

                    continue



                segments, segment_info = segment_data_from_annotations(

                    raw_clean_local,

                    annotations_df,

                    label=label

                )



                if len(segments) == 0:

                    print("  No valid segments extracted. Skipping.")

                    summary_rows.append({

                        'subject_id': subject_id,

                        'session_id': session_id,

                        'status': 'no_segments'

                    })

                    continue



                corr_mats, _ = compute_session_segment_correlation_matrices(segments)

                segment_betweenness, betweenness_df = compute_segment_betweenness_centrality(

                    corr_mats,

                    min_abs_corr=min_abs_corr,

                    use_absolute=True,

                    normalized=True

                )



                # Save outputs by subject-session ID

                out_dir = output_root / subject_id / session_id

                out_dir.mkdir(parents=True, exist_ok=True)



                gif_path = out_dir / f"{subject_session_id}_betweenness_change.gif"

                create_betweenness_change_gif(

                    segment_betweenness=segment_betweenness,

                    channel_names=list(raw_clean_local.ch_names),

                    output_path=gif_path,

                    fps=gif_fps

                )



                betweenness_df = betweenness_df.copy()

                betweenness_df['subject_id'] = subject_id

                betweenness_df['session_id'] = session_id

                betweenness_csv_path = out_dir / f"{subject_session_id}_betweenness_by_segment.csv"

                betweenness_df.to_csv(betweenness_csv_path, index=False)



                # Save matrix form: rows=segments, cols=nodes

                betweenness_matrix = np.vstack(segment_betweenness)

                npy_path = out_dir / f"{subject_session_id}_betweenness_matrix.npy"

                np.save(npy_path, betweenness_matrix)



                summary_rows.append({

                    'subject_id': subject_id,

                    'session_id': session_id,

                    'status': 'ok',

                    'n_segments': len(segments),

                    'n_nodes': betweenness_matrix.shape[1],

                    'gif_path': str(gif_path),

                    'csv_path': str(betweenness_csv_path),

                    'npy_path': str(npy_path)

                })



                print(f"  Saved: {gif_path.name}, {betweenness_csv_path.name}, {npy_path.name}")



            except Exception as exc:

                print(f"  Failed: {exc}")

                summary_rows.append({

                    'subject_id': subject_id,

                    'session_id': session_id,

                    'status': f'failed: {exc}'

                })



    summary_df = pd.DataFrame(summary_rows)

    summary_path = output_root / 'multi_session_betweenness_summary.csv'

    summary_df.to_csv(summary_path, index=False)



    print(f"\nFinished. Summary saved to: {summary_path}")

    return summary_df





# Run batch processing for subjects with more than one session

multi_session_summary = process_multi_session_subjects_betweenness(

    dataset_path=dataset_path,

    output_root='betweenness_multi_session',

    label='eyes_closed',

    min_abs_corr=0.0,

    gif_fps=2

)



print(multi_session_summary.head())

print(f"Total processed rows: {len(multi_session_summary)}")


Subjects with >1 session: 22

Processing sub-NORB00064_ses-1...

Loaded: sub-NORB00064_ses-1_task-EEG_eeg.edf

Extracted 13 valid segments
Computed 13 correlation matrix/matrices from 13 segment(s).
Computed betweenness centrality for 13 segment(s).
GIF saved to: /home/aloo/CNS_Summer_Project_1/betweenness_multi_session/sub-NORB00064/ses-1/sub-NORB00064_ses-1_betweenness_change.gif
  Saved: sub-NORB00064_ses-1_betweenness_change.gif, sub-NORB00064_ses-1_betweenness_by_segment.csv, sub-NORB00064_ses-1_betweenness_matrix.npy

Processing sub-NORB00064_ses-2...

Loaded: sub-NORB00064_ses-2_task-EEG_eeg.edf

Extracted 20 valid segments
Computed 20 correlation matrix/matrices from 20 segment(s).
Computed betweenness centrality for 20 segment(s).
GIF saved to: /home/aloo/CNS_Summer_Project_1/betweenness_multi_session/sub-NORB00064/ses-2/sub-NORB00064_ses-2_betweenness_change.gif
  Saved: sub-NORB00064_ses-2_betweenness_change.gif, sub-NORB00064_ses-2_betweenness_by_segment.csv, sub-NORB00064_